# Script 3 — Treinamento dos Modelos de ML (V5)
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

| Decisão | Justificativa |
|---------|---------------|
| **9 targets** | 3 primários (DRE) + 5 balanço + 1 caixa → habilita Z''-Score prospectivo completo |
| **Split temporal** | Treino ≤2022, Teste 2023–2024 — o modelo nunca viu dados futuros |
| **GroupKFold por empresa** | Evita vazamento temporal cruzado entre empresas no CV |
| **Transformação seletiva por target** | log1p para séries positivas e arcsinh para séries negativas/mistas |
| **SMAPE como métrica principal** | Definido para negativos (Lucro Líquido pode ser negativo) |
| **Theil's U** | Prova que ML supera baseline ingênua |
| **Acurácia Direcional** | Percentual de acertos na direção (sobe/desce) |
| **Imputer no Pipeline** | Evita leakage de NaN das features YoY no CV |
| **Curvas de aprendizado** | Diagnóstico automático de overfitting/underfitting |
| **Análise de resíduos** | Valida pressupostos e detecta padrões sistemáticos |

## Etapa 0 — Dependências e configuração

In [1]:
import logging, json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, learning_curve
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)

logger = logging.getLogger('pipeline_modelagem_v5')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                         datefmt='%Y-%m-%d %H:%M:%S')
_sh = logging.StreamHandler()
_sh.setLevel(logging.INFO)
_sh.setFormatter(_fmt)
logger.addHandler(_sh)
_fh = logging.FileHandler(PASTA_SAIDA / 'pipeline_modelagem_v5.log',
                          mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG)
_fh.setFormatter(_fmt)
logger.addHandler(_fh)

# ── Parâmetros ─────────────────────────────────────────────────────────────
N_SPLITS_EXT = 5
N_SPLITS_INT = 5
RANDOM_STATE = 42
ANO_CORTE    = 2022   # treino ≤ 2022, teste ≥ 2023

# Targets que recebem log1p (séries positivas e assimétricas)
LOG_TARGETS = {
    'TARGET_DRE_3.01', 'TARGET_EBITDA',
    'TARGET_BPA_1', 'TARGET_BPA_1.01',
    'TARGET_BPP_2.01', 'TARGET_BPP_2.03', 'TARGET_BPP_2',
}

# Targets com valores negativos ou mistos recebem arcsinh
ARCSINH_TARGETS = {
    'TARGET_DFC_MI_6.01',
    'TARGET_DRE_3.11',  # Lucro Líquido pode ser negativo
}

# Alias retrocompatível para qualquer alvo com transformação
TRANSFORM_TARGETS = LOG_TARGETS | ARCSINH_TARGETS


def get_target_transform(target):
    """
    Retorna a transformação do target:
    - 'log1p'   para séries positivas e assimétricas
    - 'arcsinh' para séries negativas ou mistas
    - 'none'    caso o target não receba transformação
    """
    if target in LOG_TARGETS:
        return 'log1p'
    if target in ARCSINH_TARGETS:
        return 'arcsinh'
    return 'none'


def target_transform(y, transformacao='none'):
    """
    Aplica a transformação escolhida ao target.

    transformacao:
        - 'log1p'   : apenas para valores > -1
        - 'arcsinh' : válida para qualquer valor real
        - 'none'    : sem transformação
    """
    y_arr = np.asarray(y, dtype=float)

    if not np.isfinite(y_arr).all():
        n_bad = np.size(y_arr) - np.isfinite(y_arr).sum()
        raise ValueError(
            f'target_transform: há {n_bad} valores não finitos antes da transformação.'
        )

    if transformacao == 'log1p':
        if np.any(y_arr <= -1):
            n_invalidos = int(np.sum(y_arr <= -1))
            raise ValueError(
                f"target_transform(log1p): {n_invalidos} valores <= -1 encontrados. "
                f"Use 'arcsinh' para targets com negativos/mistos."
            )
        return np.log1p(y_arr)

    if transformacao == 'arcsinh':
        return np.arcsinh(y_arr)

    return y_arr.copy()


def target_inverse_transform(y_pred, transformacao='none'):
    """Inverte a transformação aplicada ao target."""
    y_arr = np.asarray(y_pred, dtype=float)

    if transformacao == 'log1p':
        return np.expm1(y_arr)
    if transformacao == 'arcsinh':
        return np.sinh(y_arr)
    return y_arr


logger.info("Script 3 V5 iniciado | sklearn=%s", __import__('sklearn').__version__)
print("✅ Dependências carregadas")

2026-05-07 15:29:26 | INFO     | Script 3 V5 iniciado | sklearn=1.8.0


✅ Dependências carregadas


## Etapa 1 — Carregamento e split temporal

In [2]:
dataset = pd.read_parquet(PASTA_SAIDA / 'dataset_preparado.parquet')

with open(PASTA_SAIDA / 'features.pkl',      'rb') as f: FEATURES      = pickle.load(f)
with open(PASTA_SAIDA / 'targets.pkl',       'rb') as f: TARGETS       = pickle.load(f)
with open(PASTA_SAIDA / 'kpis.pkl',          'rb') as f: KPIS          = pickle.load(f)
with open(PASTA_SAIDA / 'grupos_treino.pkl', 'rb') as f: GRUPOS_TREINO = pickle.load(f)

# ── Carregar splits do Script 2 (já com 3 conjuntos: treino/teste/prospectivo) ──
# O Script 2 grava treino.parquet e teste.parquet com o split temporal correto.
# O Script 3 consome esses arquivos diretamente — sem recriar o split —
# para garantir que treino ≤ 2022, teste 2023–2024, prospectivo ≥ 2025.
cam_treino = PASTA_SAIDA / 'treino.parquet'
cam_teste  = PASTA_SAIDA / 'teste.parquet'

if cam_treino.exists() and cam_teste.exists():
    treino = pd.read_parquet(cam_treino)
    teste  = pd.read_parquet(cam_teste)
    logger.info("Splits carregados dos parquets do Script 2")
else:
    # Fallback: Script 2 não foi rodado — recalcular com 3 conjuntos
    logger.warning("treino.parquet não encontrado — recalculando split temporal")
    if 'split' not in dataset.columns:
        dataset['split'] = np.where(
            dataset['ANO'].astype(float) <= ANO_CORTE, 'treino',
            np.where(
                dataset['ANO'].astype(float) <= 2024, 'teste',
                'prospectivo'
            )
        )
    treino = dataset[dataset['split'] == 'treino'].reset_index(drop=True)
    teste  = dataset[dataset['split'] == 'teste'].reset_index(drop=True)
    GRUPOS_TREINO = treino['CNPJ_CIA'].values
    treino.to_parquet(cam_treino, index=False)
    teste.to_parquet(cam_teste,   index=False)
    with open(PASTA_SAIDA / 'grupos_treino.pkl', 'wb') as f:
        pickle.dump(GRUPOS_TREINO, f)

# Validação
assert len(treino) > 0, "Treino vazio — verifique o Script 2"
assert len(teste)  > 0, "Teste vazio — verifique o Script 2"

# Verificar que o conjunto prospectivo NÃO vazou para treino ou teste
anos_treino = set(treino['ANO'].dropna().astype(int).unique())
anos_teste  = set(teste['ANO'].dropna().astype(int).unique())
anos_prosp  = {a for a in anos_treino | anos_teste if a >= 2025}
if anos_prosp:
    logger.error("Anos prospectivos vazaram para treino/teste: %s", anos_prosp)
else:
    logger.info("Isolamento prospectivo: PASSOU ✅ (nenhum ano ≥2025 no treino/teste)")

n_tot = len(treino) + len(teste)
logger.info("Treino: %d obs (≤%d) | Teste: %d obs (%d–%d)",
            len(treino), ANO_CORTE,
            len(teste),  ANO_CORTE+1, 2024)

print(f"\n{'='*65}")
print(f"  Split temporal — carregado do Script 2")
print(f"{'='*65}")
print(f"  Treino (≤{ANO_CORTE}): {len(treino):>4} obs | "
      f"DFP={(treino['ORIGEM']=='DFP').sum():>3} | "
      f"ITR={(treino['ORIGEM']=='ITR').sum():>3} | "
      f"anos {sorted(anos_treino)[:3]}...{sorted(anos_treino)[-1:]}")
print(f"  Teste ({ANO_CORTE+1}–2024): {len(teste):>4} obs | "
      f"DFP={(teste['ORIGEM']=='DFP').sum():>3} | "
      f"ITR={(teste['ORIGEM']=='ITR').sum():>3} | "
      f"anos {sorted(anos_teste)}")
print(f"  Prospectivo (≥2025): carregado pelo Script 5")
print(f"  Isolamento prospectivo: ✅")
print(f"{'='*65}")
print(f"\nFeatures: {len(FEATURES)} | Targets: {len(TARGETS)}")
print(f"Targets:")
for t in TARGETS:
    if t in treino.columns:
        # FIX: indicar também targets arcsinh, não apenas log1p
        if t in LOG_TARGETS:
            transform_flag = "📐log   "
        elif t in ARCSINH_TARGETS:
            transform_flag = "📐arcsinh"
        else:
            transform_flag = "         "
        n_tr = treino[t].notna().sum()
        n_te = teste[t].notna().sum() if t in teste.columns else 0
        print(f"  {transform_flag} {t:<30} treino={n_tr:,} | teste={n_te:,}")

2026-05-07 15:29:30 | INFO     | Splits carregados dos parquets do Script 2
2026-05-07 15:29:30 | INFO     | Isolamento prospectivo: PASSOU ✅ (nenhum ano ≥2025 no treino/teste)
2026-05-07 15:29:30 | INFO     | Treino: 713 obs (≤2022) | Teste: 168 obs (2023–2024)



  Split temporal — carregado do Script 2
  Treino (≤2022):  713 obs | DFP=181 | ITR=532 | anos [np.int64(2015), np.int64(2016), np.int64(2017)]...[np.int64(2022)]
  Teste (2023–2024):  168 obs | DFP= 24 | ITR=144 | anos [np.int64(2023), np.int64(2024)]
  Prospectivo (≥2025): carregado pelo Script 5
  Isolamento prospectivo: ✅

Features: 15 | Targets: 9
Targets:
  📐log    TARGET_DRE_3.01                treino=713 | teste=168
  📐arcsinh TARGET_DRE_3.11                treino=713 | teste=168
  📐log    TARGET_EBITDA                  treino=713 | teste=168
  📐log    TARGET_BPA_1                   treino=713 | teste=168
  📐log    TARGET_BPA_1.01                treino=713 | teste=168
  📐log    TARGET_BPP_2.01                treino=713 | teste=168
  📐log    TARGET_BPP_2.03                treino=713 | teste=168
  📐log    TARGET_BPP_2                   treino=713 | teste=168
  📐arcsinh TARGET_DFC_MI_6.01             treino=713 | teste=168


## Etapa 2 — Métricas e Baseline Ingênua

Além das métricas estatísticas clássicas, o pipeline calcula:

**Theil's U:** U<1 prova que o modelo supera a baseline de persistência.
**Acurácia Direcional:** % de acertos na direção (sobe/desce) — mais relevante para gestores.
**SMAPE:** erro percentual simétrico, definido mesmo para valores negativos (Lucro Líquido).

In [3]:
def smape(y_true, y_pred):
    """Symmetric MAPE — definido para valores negativos e próximos de zero."""
    num   = np.abs(y_true - y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask  = denom > 0
    return float(np.mean(num[mask] / denom[mask])) if mask.sum() > 0 else np.nan


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def theil_u(y_true, y_pred):
    """
    Coeficiente de desigualdade de Theil (U2).
    U < 1 → modelo supera a baseline ingênua de persistência.
    U = 1 → equivalente à baseline.
    U > 1 → pior que não fazer nada.
    """
    n = len(y_true)
    if n < 2:
        return np.nan
    erro_modelo   = np.sqrt(np.mean((y_true[1:] - y_pred[1:])**2))
    erro_baseline = np.sqrt(np.mean((y_true[1:] - y_true[:-1])**2))
    return float(erro_modelo / erro_baseline) if erro_baseline > 0 else np.nan


def acuracia_direcional(y_true, y_pred):
    """
    Percentual de acertos na direção (sobe/desce) em relação ao período anterior.
    Calculado sobre sequências ordenadas temporalmente.
    """
    if len(y_true) < 2:
        return np.nan
    dir_real = np.sign(np.diff(y_true))
    dir_pred = np.sign(np.diff(y_pred))
    mask = dir_real != 0
    return float(np.mean(dir_real[mask] == dir_pred[mask])) if mask.sum() > 0 else np.nan


def calcular_baseline(treino_df, teste_df, target):
    """
    Baseline ingênua por empresa: persistência do último valor observado
    da própria companhia, respeitando a ordem temporal.

    A previsão de cada linha do teste usa o último valor não-nulo do target
    para a mesma empresa, vindo do histórico do treino ou de períodos anteriores
    do próprio teste. Isso evita mistura entre companhias diferentes e produz
    uma comparação mais justa do que uma baseline global.

    Detecção automática da coluna temporal (prioridade decrescente):
        DT_FIM_EXERC → DATA_REFERENCIA → DATA → DT_REFERENCIA →
        TRIMESTRE → TRI → PERIODO → PERÍODO → ANO

    Se nenhuma coluna temporal estiver disponível, a ordem original é preservada
    dentro de cada empresa (comportamento idêntico a usar ANO como fallback).

    Linhas do teste sem histórico disponível são excluídas da avaliação;
    a métrica de cobertura indica a fração do teste avaliada.
    """
    if target not in treino_df.columns or target not in teste_df.columns:
        return {}

    if 'CNPJ_CIA' not in treino_df.columns or 'CNPJ_CIA' not in teste_df.columns:
        return {}

    # Detecta a coluna temporal mais específica disponível
    candidatos_tempo = [
        'DT_FIM_EXERC', 'DATA_REFERENCIA', 'DATA', 'DT_REFERENCIA',
        'TRIMESTRE', 'TRI', 'PERIODO', 'PERÍODO', 'ANO'
    ]
    time_col = next(
        (c for c in candidatos_tempo
         if c in treino_df.columns and c in teste_df.columns),
        None
    )

    # Colunas de ordenação: empresa + tempo + posição original (desempate estável)
    cols_ordenacao = ['CNPJ_CIA']
    if time_col is not None:
        cols_ordenacao.append(time_col)

    # Preserva a posição original como desempate para garantir estabilidade
    treino_tmp = treino_df.reset_index(drop=True).copy()
    teste_tmp  = teste_df.reset_index(drop=True).copy()
    treino_tmp['_ordem_original'] = np.arange(len(treino_tmp))
    teste_tmp['_ordem_original']  = np.arange(len(teste_tmp))

    cols_select = list(dict.fromkeys(cols_ordenacao + ['_ordem_original', target]))

    base = pd.concat([
        treino_tmp[cols_select].assign(__split='treino'),
        teste_tmp[cols_select].assign(__split='teste'),
    ], ignore_index=True)

    # Ordena dentro de cada empresa para construir a persistência cronológica
    base = base.sort_values(
        cols_ordenacao + ['_ordem_original'], kind='mergesort'
    ).reset_index(drop=True)

    # Último valor observado não-nulo da própria empresa — excluindo o ponto atual
    base['baseline_prev'] = (
        base.groupby('CNPJ_CIA')[target]
            .transform(lambda s: s.ffill().shift(1))
    )

    # Avalia somente o conjunto de teste onde há baseline disponível
    mask_teste   = base['__split'] == 'teste'
    mask_valido  = mask_teste & base[target].notna() & base['baseline_prev'].notna()

    if mask_valido.sum() == 0:
        return {}

    y_t = base.loc[mask_valido, target].values
    y_p = base.loc[mask_valido, 'baseline_prev'].values

    n_teste_total = int(mask_teste.sum())
    cobertura = float(mask_valido.sum() / max(1, n_teste_total))

    return {
        'RMSE_baseline'        : rmse(y_t, y_p),
        'MAE_baseline'         : float(mean_absolute_error(y_t, y_p)),
        'SMAPE_baseline'       : smape(y_t, y_p),
        'R2_baseline'          : float(r2_score(y_t, y_p)),
        'TheilU_baseline'      : 1.0,   # persistência é a referência por definição
        'DA_baseline'          : float(acuracia_direcional(y_t, y_p)),
        'Cobertura_baseline'   : cobertura,
        'TimeCol_baseline'     : time_col if time_col is not None else '',
    }


# Calcular baselines para todos os targets
baselines = {}
print("=== Baseline Ingênua por empresa (persistência) ===")
print(f"  {'Target':<30} {'RMSE':>14} {'SMAPE':>7} {'R²':>6} "
      f"{'TheilU':>7} {'DA':>6} {'Cob.':>6} {'ColTempo'}")
print(f"  {'-'*30} {'-'*14} {'-'*7} {'-'*6} {'-'*7} {'-'*6} {'-'*6} {'-'*14}")
for t in TARGETS:
    b = calcular_baseline(treino, teste, t)
    baselines[t] = b
    if b:
        print(f"  {t:<30} {b['RMSE_baseline']:>14,.0f} "
              f"{b['SMAPE_baseline']:>7.1%} {b['R2_baseline']:>6.3f} "
              f"{b['TheilU_baseline']:>7.2f} {b['DA_baseline']:>6.1%} "
              f"{b.get('Cobertura_baseline', np.nan):>6.1%} "
              f"  {b.get('TimeCol_baseline', 'N/A')}")
    else:
        print(f"  {t:<30} {'N/A':>14} {'N/A':>7} {'N/A':>6} "
              f"{'N/A':>7} {'N/A':>6} {'N/A':>6}  N/A")

=== Baseline Ingênua por empresa (persistência) ===
  Target                                   RMSE   SMAPE     R²  TheilU     DA   Cob. ColTempo
  ------------------------------ -------------- ------- ------ ------- ------ ------ --------------
  TARGET_DRE_3.01                    22,487,246   17.6%  0.959    1.00  59.7%  85.7%   TRIMESTRE
  TARGET_DRE_3.11                    21,816,677   71.2% -0.689    1.00  74.8%  85.7%   TRIMESTRE
  TARGET_EBITDA                       6,991,551   21.5%  0.807    1.00  64.3%  85.7%   TRIMESTRE
  TARGET_BPA_1                       23,200,605   17.7%  0.990    1.00  89.1%  85.7%   TRIMESTRE
  TARGET_BPA_1.01                     5,357,315   18.7%  0.971    1.00  61.3%  85.7%   TRIMESTRE
  TARGET_BPP_2.01                     6,304,194   25.5%  0.973    1.00  66.4%  85.7%   TRIMESTRE
  TARGET_BPP_2.03                     6,531,550   22.1%  0.993    1.00  74.8%  85.7%   TRIMESTRE
  TARGET_BPP_2                       23,200,605   17.7%  0.990    1.00  89.

## Etapa 3 — Algoritmos e grades de hiperparâmetros

In [4]:
# GroupKFold por empresa
gkf_ext = GroupKFold(n_splits=N_SPLITS_EXT)
gkf_int = GroupKFold(n_splits=N_SPLITS_INT)

# ── Ridge ──────────────────────────────────────────────────────────────────
est_ridge = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("ridge",   Ridge()),
])
grade_ridge = {"ridge__alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}

# ── SVR ────────────────────────────────────────────────────────────────────
est_svr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("svr",     SVR(kernel="rbf", max_iter=20000)),
])
grade_svr = {
    "svr__C":       [0.1, 1.0, 10.0, 100.0],
    "svr__epsilon": [0.01, 0.05, 0.1, 0.5],
    "svr__gamma":   ["scale", "auto"],
}

# ── Random Forest ──────────────────────────────────────────────────────────
est_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf",      RandomForestRegressor(n_estimators=300,
                                       random_state=RANDOM_STATE, n_jobs=-1)),
])
grade_rf = {
    "rf__max_depth":        [None, 5, 10, 20],
    "rf__min_samples_leaf": [1, 2, 5],
    "rf__max_features":     ["sqrt", "log2"],
}

# ── Gradient Boosting ──────────────────────────────────────────────────────
est_gb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("gb",      GradientBoostingRegressor(random_state=RANDOM_STATE)),
])
grade_gb = {
    "gb__n_estimators":  [100, 200, 300],
    "gb__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "gb__max_depth":     [3, 5],
    "gb__subsample":     [0.7, 0.8, 1.0],
}

ALGORITMOS = {
    "Ridge":            (est_ridge, grade_ridge),
    "SVR":              (est_svr,   grade_svr),
    "RandomForest":     (est_rf,    grade_rf),
    "GradientBoosting": (est_gb,    grade_gb),
}
logger.info("%d algoritmos | GroupKFold ext=%d int=%d",
            len(ALGORITMOS), N_SPLITS_EXT, N_SPLITS_INT)
print(f"✅ {len(ALGORITMOS)} algoritmos com GroupKFold(n={N_SPLITS_EXT})")

2026-05-07 15:29:43 | INFO     | 4 algoritmos | GroupKFold ext=5 int=5


✅ 4 algoritmos com GroupKFold(n=5)


## Etapa 4 — Treinamento com Nested Cross-Validation

In [5]:
def treinar_alg(nome, estimador, grade, X, y, grupos, gkf_int, gkf_ext,
                transformacao='none'):
    """
    Nested CV com GroupKFold.

    Loop interno : GridSearchCV seleciona hiperparâmetros sem vazar empresas.
    Loop externo : estima generalização no espaço original (transformação revertida).

    Transformação aplicada ao target:
        - log1p   para séries positivas e assimétricas
        - arcsinh para séries negativas/mistas
        - none    sem transformação

    Métricas calculadas: RMSE, MAE, SMAPE, R², Theil's U, Acurácia Direcional.
    """
    y_fit = target_transform(y, transformacao)

    # ── Loop interno ────────────────────────────────────────────────────
    gs = GridSearchCV(
        estimador, grade,
        cv=list(gkf_int.split(X, y_fit, grupos)),
        scoring='neg_mean_squared_error',
        refit=True, n_jobs=-1, verbose=0,
    )
    gs.fit(X, y_fit)
    melhor = gs.best_estimator_

    # ── Loop externo ────────────────────────────────────────────────────
    rmse_v, mae_v, smape_v, r2_v, theil_v, da_v = [], [], [], [], [], []

    for tr_idx, val_idx in gkf_ext.split(X, y_fit, grupos):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr        = y_fit[tr_idx]
        y_orig_val  = y[val_idx]

        melhor.fit(X_tr, y_tr)
        y_pred_raw = melhor.predict(X_val)
        y_pred     = target_inverse_transform(y_pred_raw, transformacao)

        rmse_v.append(rmse(y_orig_val, y_pred))
        mae_v.append(float(mean_absolute_error(y_orig_val, y_pred)))
        smape_v.append(smape(y_orig_val, y_pred))
        r2_v.append(float(r2_score(y_orig_val, y_pred)))
        theil_v.append(theil_u(y_orig_val, y_pred))
        da_v.append(acuracia_direcional(y_orig_val, y_pred))

    # Fit final no conjunto completo de treino
    melhor.fit(X, y_fit)

    def _m(lst): return float(np.nanmean(lst))
    def _s(lst): return float(np.nanstd(lst))

    metricas = {
        'RMSE_CV'      : _m(rmse_v),   'RMSE_CV_std'  : _s(rmse_v),
        'MAE_CV'       : _m(mae_v),
        'SMAPE_CV'     : _m(smape_v),  'SMAPE_CV_std' : _s(smape_v),
        'R2_CV'        : _m(r2_v),     'R2_CV_std'    : _s(r2_v),
        'TheilU_CV'    : _m(theil_v),
        'DA_CV'        : _m(da_v),
        'transformacao': transformacao,
        'log_transform': transformacao == 'log1p',
        'best_params'  : gs.best_params_,
    }

    flag_theil = "✅" if metricas['TheilU_CV'] < 1 else "⚠️"
    logger.info("  %-20s RMSE=%10.0f±%8.0f  SMAPE=%5.1f%%  "
                "R²=%5.3f  TheilU=%s%.3f  DA=%.1f%%  transf=%s",
                nome, metricas['RMSE_CV'], metricas['RMSE_CV_std'],
                metricas['SMAPE_CV']*100, metricas['R2_CV'],
                flag_theil, metricas['TheilU_CV'],
                metricas['DA_CV']*100, transformacao)
    print(f"  {flag_theil} {nome:<20} "
          f"RMSE={metricas['RMSE_CV']:>12,.0f}  "
          f"SMAPE={metricas['SMAPE_CV']:>5.1%}  "
          f"R²={metricas['R2_CV']:>6.3f}  "
          f"U={metricas['TheilU_CV']:.3f}  "
          f"DA={metricas['DA_CV']:.1%}")

    return melhor, metricas


# ── Execução ──────────────────────────────────────────────────────────────
resultados = {}

for target in TARGETS:
    # FIX: derivar 'transformacao' via get_target_transform — variável estava
    # sendo usada na chamada a treinar_alg sem ter sido definida no loop.
    transformacao = get_target_transform(target)
    b = baselines.get(target, {})

    print(f"\n{'='*72}")
    print(f"  TARGET: {target}  |  transform={transformacao}")
    if b:
        print(f"  Baseline → RMSE={b.get('RMSE_baseline', 0):,.0f}  "
              f"SMAPE={b.get('SMAPE_baseline', 0):.1%}  "
              f"R²={b.get('R2_baseline', 0):.3f}  "
              f"DA={b.get('DA_baseline', 0):.1%}  "
              f"Cob.={b.get('Cobertura_baseline', np.nan):.1%}")
    print(f"  {'Alg':<22} {'RMSE':>14} {'SMAPE':>7} {'R²':>7} {'TheilU':>7} {'DA':>6}")
    print(f"  {'-'*22} {'-'*14} {'-'*7} {'-'*7} {'-'*7} {'-'*6}")

    # Filtrar obs com target não-nulo
    df_t = treino[FEATURES + [target]].copy()
    df_t = df_t[df_t[target].notna()].reset_index(drop=True)
    mask = treino[target].notna()
    grupos_t = GRUPOS_TREINO[mask.values]

    X = df_t[FEATURES].values
    y = df_t[target].values

    resultados[target] = {}
    for nome, (est, grade) in ALGORITMOS.items():
        modelo, metricas = treinar_alg(
            nome, est, grade, X, y, grupos_t,
            gkf_int, gkf_ext, transformacao=transformacao,
        )
        resultados[target][nome] = (modelo, metricas)
        joblib.dump(
            {'modelo': modelo, 'transformacao': transformacao,
             'log_transform': transformacao == 'log1p', 'features': FEATURES},
            PASTA_SAIDA / f'modelo_{target}_{nome}.pkl'
        )

    logger.info("TARGET %s concluído", target)
print("\n✅ Treinamento concluído para todos os targets.")


  TARGET: TARGET_DRE_3.01  |  transform=log1p
  Baseline → RMSE=22,487,246  SMAPE=17.6%  R²=0.959  DA=59.7%  Cob.=85.7%
  Alg                              RMSE   SMAPE      R²  TheilU     DA
  ---------------------- -------------- ------- ------- ------- ------


2026-05-07 15:29:55 | INFO     |   Ridge                RMSE=10977485992±21875867874  SMAPE= 73.3%  R²=-22235.551  TheilU=⚠️196.116  DA=92.8%  transf=log1p


  ⚠️ Ridge                RMSE=10,977,485,992  SMAPE=73.3%  R²=-22235.551  U=196.116  DA=92.8%


2026-05-07 15:29:56 | INFO     |   SVR                  RMSE=  43674502±58329042  SMAPE= 33.9%  R²=0.697  TheilU=⚠️1.393  DA=91.8%  transf=log1p


  ⚠️ SVR                  RMSE=  43,674,502  SMAPE=33.9%  R²= 0.697  U=1.393  DA=91.8%


...

  ⚠️ SVR                  RMSE=  20,293,488  SMAPE=71.3%  R²= 0.246  U=2.043  DA=57.0%


2026-05-07 15:38:38 | INFO     |   RandomForest         RMSE=  11445243±13092106  SMAPE= 97.6%  R²=0.520  TheilU=⚠️1.488  DA=49.9%  transf=arcsinh


  ⚠️ RandomForest         RMSE=  11,445,243  SMAPE=97.6%  R²= 0.520  U=1.488  DA=49.9%


2026-05-07 15:39:16 | INFO     |   GradientBoosting     RMSE=  17342017±23272715  SMAPE= 68.9%  R²=0.460  TheilU=⚠️1.687  DA=33.6%  transf=arcsinh
2026-05-07 15:39:16 | INFO     | TARGET TARGET_DFC_MI_6.01 concluído


  ⚠️ GradientBoosting     RMSE=  17,342,017  SMAPE=68.9%  R²= 0.460  U=1.687  DA=33.6%

✅ Treinamento concluído para todos os targets.


## Etapa 5 — Avaliação no conjunto de teste hold-out

In [6]:
def avaliar_teste(modelo, X_te, y_te, transformacao):
    y_pred_raw = modelo.predict(X_te)
    y_pred = target_inverse_transform(y_pred_raw, transformacao)
    mask = np.isfinite(y_te) & np.isfinite(y_pred)
    yt, yp = y_te[mask], y_pred[mask]
    return {
        'RMSE_teste'  : rmse(yt, yp),
        'MAE_teste'   : float(mean_absolute_error(yt, yp)),
        'SMAPE_teste' : smape(yt, yp),
        'R2_teste'    : float(r2_score(yt, yp)),
        'TheilU_teste': theil_u(yt, yp),
        'DA_teste'    : acuracia_direcional(yt, yp),
    }


print("\n=== Avaliação no Teste Hold-out (2023–2024) ===")
metricas_teste = {}

for target in TARGETS:
    transformacao = get_target_transform(target)
    b = baselines.get(target, {})
    df_te = teste[FEATURES + [target]].copy()
    df_te = df_te[df_te[target].notna()]
    X_te  = df_te[FEATURES].values
    y_te  = df_te[target].values

    metricas_teste[target] = {}
    baseline_rmse = b.get('RMSE_baseline', np.inf)

    print(f"\n{target}  (baseline RMSE={baseline_rmse:,.0f}  "
          f"DA={b.get('DA_baseline', 0):.1%}  "
          f"Cob.={b.get('Cobertura_baseline', np.nan):.1%})")
    print(f"  {'Algoritmo':<20} {'RMSE':>14} {'SMAPE':>7} "
          f"{'R²':>7} {'TheilU':>7} {'DA':>6} {'Bateu?':>7}")
    print(f"  {'-'*20} {'-'*14} {'-'*7} {'-'*7} {'-'*7} {'-'*6} {'-'*7}")

    for nome, (modelo, _) in resultados[target].items():
        m = avaliar_teste(modelo, X_te, y_te, transformacao)
        metricas_teste[target][nome] = m
        bateu    = m['RMSE_teste'] < baseline_rmse
        theil_ok = (m['TheilU_teste'] or 1.0) < 1.0
        flag = "✅" if bateu and theil_ok else ("🟡" if bateu else "❌")
        print(f"  {flag} {nome:<18} {m['RMSE_teste']:>14,.0f} "
              f"{m['SMAPE_teste']:>7.1%} {m['R2_teste']:>7.3f} "
              f"{m['TheilU_teste']:>7.3f} {m['DA_teste']:>6.1%} "
              f"{'✅' if bateu else '❌':>7}")
        logger.info("Teste | %s | %s: RMSE=%.0f SMAPE=%.2f%% R2=%.3f TheilU=%.3f DA=%.1f%%",
                    target, nome, m['RMSE_teste'], m['SMAPE_teste']*100,
                    m['R2_teste'], m['TheilU_teste'], m['DA_teste']*100)

2026-05-07 15:40:42 | INFO     | Teste | TARGET_DRE_3.01 | Ridge: RMSE=756593961 SMAPE=54.06% R2=-44.868 TheilU=13.757 DA=91.5%
2026-05-07 15:40:42 | INFO     | Teste | TARGET_DRE_3.01 | SVR: RMSE=61370606 SMAPE=19.86% R2=0.698 TheilU=1.116 DA=95.7%
2026-05-07 15:40:42 | INFO     | Teste | TARGET_DRE_3.01 | RandomForest: RMSE=13204913 SMAPE=7.72% R2=0.986 TheilU=0.240 DA=89.4%
2026-05-07 15:40:42 | INFO     | Teste | TARGET_DRE_3.01 | GradientBoosting: RMSE=6859573 SMAPE=3.43% R2=0.996 TheilU=0.125 DA=100.0%
2026-05-07 15:40:42 | INFO     | Teste | TARGET_DRE_3.11 | Ridge: RMSE=103454402 SMAPE=170.79% R2=-30.872 TheilU=8.037 DA=76.6%
2026-05-07 15:40:42 | INFO     | Teste | TARGET_DRE_3.11 | SVR: RMSE=58283758 SMAPE=88.65% R2=-9.116 TheilU=4.528 DA=70.2%



=== Avaliação no Teste Hold-out (2023–2024) ===

TARGET_DRE_3.01  (baseline RMSE=22,487,246  DA=59.7%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                 756,593,961   54.1% -44.868  13.757  91.5%       ❌
  ❌ SVR                    61,370,606   19.9%   0.698   1.116  95.7%       ❌
  ✅ RandomForest           13,204,913    7.7%   0.986   0.240  89.4%       ✅
  ✅ GradientBoosting        6,859,573    3.4%   0.996   0.125 100.0%       ✅

TARGET_DRE_3.11  (baseline RMSE=21,816,677  DA=74.8%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                 103,454,402  170.8% -30.872   8.037  76.6%       ❌
  ❌ SVR                    58,283,758   88.7%  -9.116   4.528  70.2%       ❌
  ✅ RandomForest            7,649,942   71.0%  

2026-05-07 15:40:42 | INFO     | Teste | TARGET_DRE_3.11 | RandomForest: RMSE=7649942 SMAPE=70.97% R2=0.826 TheilU=0.594 DA=68.1%
2026-05-07 15:40:42 | INFO     | Teste | TARGET_DRE_3.11 | GradientBoosting: RMSE=14571436 SMAPE=69.26% R2=0.368 TheilU=1.132 DA=59.6%
2026-05-07 15:40:42 | INFO     | Teste | TARGET_EBITDA | Ridge: RMSE=27723505 SMAPE=86.85% R2=-2.018 TheilU=4.068 DA=83.3%
2026-05-07 15:40:42 | INFO     | Teste | TARGET_EBITDA | SVR: RMSE=19070791 SMAPE=29.04% R2=-0.428 TheilU=2.806 DA=85.7%
2026-05-07 15:40:42 | INFO     | Teste | TARGET_EBITDA | RandomForest: RMSE=6085051 SMAPE=22.07% R2=0.855 TheilU=0.895 DA=88.1%
2026-05-07 15:40:42 | INFO     | Teste | TARGET_EBITDA | GradientBoosting: RMSE=6382429 SMAPE=24.67% R2=0.840 TheilU=0.938 DA=85.7%
2026-05-07 15:40:42 | INFO     | Teste | TARGET_BPA_1 | Ridge: RMSE=625871777 SMAPE=78.96% R2=-6.687 TheilU=5.141 DA=78.7%
2026-05-07 15:40:42 | INFO     | Teste | TARGET_BPA_1 | SVR: RMSE=138640740 SMAPE=24.09% R2=0.623 TheilU=1.1

  🟡 GradientBoosting       14,571,436   69.3%   0.368   1.132  59.6%       ✅

TARGET_EBITDA  (baseline RMSE=6,991,551  DA=64.3%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                  27,723,505   86.8%  -2.018   4.068  83.3%       ❌
  ❌ SVR                    19,070,791   29.0%  -0.428   2.806  85.7%       ❌
  ✅ RandomForest            6,085,051   22.1%   0.855   0.895  88.1%       ✅
  ✅ GradientBoosting        6,382,429   24.7%   0.840   0.938  85.7%       ✅

TARGET_BPA_1  (baseline RMSE=23,200,605  DA=89.1%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                 625,871,777   79.0%  -6.687   5.141  78.7%       ❌
  ❌ SVR                   138,640,740   24.1%   0.623   1.139  76.6%       ❌
  ❌ RandomForest         

2026-05-07 15:40:43 | INFO     | Teste | TARGET_BPA_1.01 | RandomForest: RMSE=5251112 SMAPE=11.76% R2=0.973 TheilU=0.328 DA=74.5%
2026-05-07 15:40:43 | INFO     | Teste | TARGET_BPA_1.01 | GradientBoosting: RMSE=4560400 SMAPE=6.23% R2=0.980 TheilU=0.285 DA=76.6%
2026-05-07 15:40:43 | INFO     | Teste | TARGET_BPP_2.01 | Ridge: RMSE=63217523 SMAPE=55.22% R2=-1.803 TheilU=3.220 DA=66.0%
2026-05-07 15:40:43 | INFO     | Teste | TARGET_BPP_2.01 | SVR: RMSE=25742844 SMAPE=25.14% R2=0.535 TheilU=1.311 DA=74.5%
2026-05-07 15:40:43 | INFO     | Teste | TARGET_BPP_2.01 | RandomForest: RMSE=7387899 SMAPE=11.13% R2=0.962 TheilU=0.376 DA=76.6%
2026-05-07 15:40:43 | INFO     | Teste | TARGET_BPP_2.01 | GradientBoosting: RMSE=6583418 SMAPE=8.26% R2=0.970 TheilU=0.335 DA=80.9%
2026-05-07 15:40:43 | INFO     | Teste | TARGET_BPP_2.03 | Ridge: RMSE=64832928 SMAPE=60.03% R2=0.352 TheilU=1.473 DA=78.7%
2026-05-07 15:40:43 | INFO     | Teste | TARGET_BPP_2.03 | SVR: RMSE=42418622 SMAPE=28.22% R2=0.723 The

  ✅ RandomForest            5,251,112   11.8%   0.973   0.328  74.5%       ✅
  ✅ GradientBoosting        4,560,400    6.2%   0.980   0.285  76.6%       ✅

TARGET_BPP_2.01  (baseline RMSE=6,304,194  DA=66.4%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                  63,217,523   55.2%  -1.803   3.220  66.0%       ❌
  ❌ SVR                    25,742,844   25.1%   0.535   1.311  74.5%       ❌
  ❌ RandomForest            7,387,899   11.1%   0.962   0.376  76.6%       ❌
  ❌ GradientBoosting        6,583,418    8.3%   0.970   0.335  80.9%       ❌

TARGET_BPP_2.03  (baseline RMSE=6,531,550  DA=74.8%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                  64,832,928   60.0%   0.352   1.473  78.7%       ❌
  ❌ SVR              

2026-05-07 15:40:43 | INFO     | Teste | TARGET_BPP_2 | RandomForest: RMSE=27922601 SMAPE=11.16% R2=0.985 TheilU=0.229 DA=87.2%
2026-05-07 15:40:43 | INFO     | Teste | TARGET_BPP_2 | GradientBoosting: RMSE=34941625 SMAPE=7.92% R2=0.976 TheilU=0.287 DA=91.5%
2026-05-07 15:40:43 | INFO     | Teste | TARGET_DFC_MI_6.01 | Ridge: RMSE=721585009 SMAPE=159.07% R2=-295.558 TheilU=30.850 DA=66.0%
2026-05-07 15:40:43 | INFO     | Teste | TARGET_DFC_MI_6.01 | SVR: RMSE=42525578 SMAPE=65.82% R2=-0.030 TheilU=1.818 DA=66.0%
2026-05-07 15:40:43 | INFO     | Teste | TARGET_DFC_MI_6.01 | RandomForest: RMSE=6596882 SMAPE=75.26% R2=0.975 TheilU=0.282 DA=68.1%
2026-05-07 15:40:43 | INFO     | Teste | TARGET_DFC_MI_6.01 | GradientBoosting: RMSE=30595624 SMAPE=36.84% R2=0.467 TheilU=1.308 DA=66.0%


  ❌ RandomForest           27,922,601   11.2%   0.985   0.229  87.2%       ❌
  ❌ GradientBoosting       34,941,625    7.9%   0.976   0.287  91.5%       ❌

TARGET_DFC_MI_6.01  (baseline RMSE=8,086,660  DA=51.3%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                 721,585,009  159.1% -295.558  30.850  66.0%       ❌
  ❌ SVR                    42,525,578   65.8%  -0.030   1.818  66.0%       ❌
  ✅ RandomForest            6,596,882   75.3%   0.975   0.282  68.1%       ✅
  ❌ GradientBoosting       30,595,624   36.8%   0.467   1.308  66.0%       ❌


## Etapa 5B - Seleção do melhor modelo

In [7]:
def escolher_melhor_modelo_cv(resultados_target):
    """
    Escolhe o melhor algoritmo usando apenas as métricas de CV.
    Critério principal: menor RMSE_CV.
    Critérios de desempate: menor TheilU_CV e menor SMAPE_CV.
    """
    return min(
        resultados_target.items(),
        key=lambda item: (
            item[1][1].get('RMSE_CV', np.inf),
            item[1][1].get('TheilU_CV', np.inf),
            item[1][1].get('SMAPE_CV', np.inf),
        )
    )[0]

melhores = {t: escolher_melhor_modelo_cv(resultados[t]) for t in TARGETS}

## Etapa 6 — Feature Importance

In [8]:
def extrair_importancia(modelo, features, nome_alg):
    step = [s for s, _ in modelo.steps][-1]
    est_final = modelo.named_steps[step]
    if hasattr(est_final, 'feature_importances_'):
        imp = est_final.feature_importances_
    elif hasattr(est_final, 'coef_'):
        imp = np.abs(est_final.coef_)
    else:
        return pd.Series(dtype=float)
    return pd.Series(imp, index=features).sort_values(ascending=False)


feature_importances = {}
print("\n=== Feature Importance — Melhor Modelo por Target (critério: R²) ===")

n_t = len(TARGETS)
fig, axes = plt.subplots(n_t, 1, figsize=(11, 5*n_t))
if n_t == 1: axes = [axes]

for i, target in enumerate(TARGETS):
    melhor_nome = melhores[target]
    melhor_mod  = resultados[target][melhor_nome][0]
    imp = extrair_importancia(melhor_mod, FEATURES, melhor_nome)
    feature_importances[target] = {'algoritmo': melhor_nome,
                                    'importancias': imp.to_dict()}

    if not imp.empty:
        top = imp.head(min(12, len(imp)))
        colors = ['#1f4e79' if v == top.values[0] else
                  '#2e75b6' if v >= top.values[0]*0.7 else '#9dc3e6'
                  for v in top.values[::-1]]
        axes[i].barh(range(len(top)), top.values[::-1], color=colors, alpha=0.9)
        axes[i].set_yticks(range(len(top)))
        axes[i].set_yticklabels(top.index[::-1], fontsize=9)
        axes[i].set_title(f"{target.replace('TARGET_', '')} — {melhor_nome} "
                           f"(R²={metricas_teste[target][melhor_nome]['R2_teste']:.3f})",
                           fontsize=11, fontweight='bold')
        axes[i].set_xlabel('Importância Relativa')
        axes[i].grid(axis='x', alpha=0.3)
        for j, v in enumerate(top.values[::-1]):
            axes[i].text(v + imp.max()*0.005, j, f'{v:.3f}', va='center', fontsize=8)
    print(f"  {target}: {melhor_nome} | top3={list(imp.head(3).index)}")

plt.suptitle('Feature Importance — Melhor Modelo por Target',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Salvo: feature_importance.png")


=== Feature Importance — Melhor Modelo por Target (critério: R²) ===
  TARGET_DRE_3.01: GradientBoosting | top3=['TARGET_DRE_3.01_lag1', 'TARGET_BPA_1.01_lag1', 'TARGET_DRE_3.01_diff1']
  TARGET_DRE_3.11: RandomForest | top3=['TARGET_DRE_3.11_lag1', 'TARGET_DRE_3.11_lag2', 'TARGET_DRE_3.11_roll4_mean']
  TARGET_EBITDA: GradientBoosting | top3=['TARGET_BPA_1_lag1', 'TARGET_BPP_2_lag1', 'TARGET_DFC_MI_6.01_lag1']
  TARGET_BPA_1: GradientBoosting | top3=['TARGET_BPP_2_lag1', 'TARGET_BPA_1_lag1', 'TARGET_BPA_1.01_lag1']
  TARGET_BPA_1.01: GradientBoosting | top3=['TARGET_BPA_1.01_lag1', 'TARGET_BPP_2.01_lag1', 'TARGET_BPA_1_lag1']
  TARGET_BPP_2.01: GradientBoosting | top3=['TARGET_BPP_2.01_lag1', 'TARGET_BPA_1.01_lag1', 'TARGET_BPA_1_lag1']
  TARGET_BPP_2.03: GradientBoosting | top3=['TARGET_BPA_1_lag1', 'TARGET_BPP_2_lag1', 'TARGET_BPP_2.03_lag2']
  TARGET_BPP_2: GradientBoosting | top3=['TARGET_BPP_2_lag1', 'TARGET_BPA_1_lag1', 'TARGET_BPA_1.01_lag1']
  TARGET_DFC_MI_6.01: RandomForest

## Etapa 7 — Curvas de Aprendizado

In [9]:
print("\nGerando curvas de aprendizado...")

n_t = len(TARGETS)
fig, axes = plt.subplots(1, n_t, figsize=(7*n_t, 5))
if n_t == 1: axes = [axes]

for i, target in enumerate(TARGETS):
    transformacao = get_target_transform(target)
    df_t = treino[FEATURES + [target]].copy()
    df_t = df_t[df_t[target].notna()].reset_index(drop=True)
    mask = treino[target].notna()
    grupos_t = GRUPOS_TREINO[mask.values]
    X = df_t[FEATURES].values
    y = target_transform(df_t[target].values, transformacao)

    # Curvas de aprendizado usam o melhor modelo pelo mesmo critério da Etapa 6 (R²)
    melhor_nome = melhores[target]
    melhor_mod  = resultados[target][melhor_nome][0]

    try:
        sizes, tr_sc, val_sc = learning_curve(
            melhor_mod, X, y,
            cv=list(gkf_ext.split(X, y, grupos_t)),
            scoring='r2',
            train_sizes=np.linspace(0.2, 1.0, 6),
            n_jobs=-1,
        )
        ax = axes[i]
        ax.plot(sizes, tr_sc.mean(1), 'o-', label='Treino',     color='#1f4e79', lw=2)
        ax.fill_between(sizes, tr_sc.mean(1)-tr_sc.std(1),
                         tr_sc.mean(1)+tr_sc.std(1), alpha=0.12, color='#1f4e79')
        ax.plot(sizes, val_sc.mean(1), 's--', label='Validação', color='#c0392b', lw=2)
        ax.fill_between(sizes, val_sc.mean(1)-val_sc.std(1),
                         val_sc.mean(1)+val_sc.std(1), alpha=0.12, color='#c0392b')
        gap  = tr_sc.mean(1)[-1] - val_sc.mean(1)[-1]
        diag = ('overfitting'  if gap > 0.15 else
                'underfitting' if val_sc.mean(1)[-1] < 0.3 else 'OK')
        ax.set_title(f"{target.replace('TARGET_', '')}\n{melhor_nome}",
                      fontsize=10, fontweight='bold')
        ax.set_xlabel(f'Tamanho do treino  |  Gap={gap:.2f} → {diag}', fontsize=9)
        ax.set_ylabel('R²')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
        logger.info("Curva %s/%s: gap=%.3f diag=%s", target, melhor_nome, gap, diag)
    except Exception as e:
        logger.warning("Curva de aprendizado falhou %s/%s: %s", target, melhor_nome, e)
        axes[i].text(0.5, 0.5, 'Erro na curva\n'+str(e)[:60],
                     ha='center', va='center', transform=axes[i].transAxes, fontsize=9)

plt.suptitle('Curvas de Aprendizado — Diagnóstico de Bias/Variância',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'curvas_aprendizado.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Salvo: curvas_aprendizado.png")


Gerando curvas de aprendizado...


2026-05-07 15:43:48 | INFO     | Curva TARGET_DRE_3.01/GradientBoosting: gap=0.042 diag=OK
2026-05-07 15:43:52 | INFO     | Curva TARGET_DRE_3.11/RandomForest: gap=1.097 diag=overfitting
2026-05-07 15:43:53 | INFO     | Curva TARGET_EBITDA/GradientBoosting: gap=0.094 diag=OK
2026-05-07 15:43:56 | INFO     | Curva TARGET_BPA_1/GradientBoosting: gap=0.037 diag=OK
2026-05-07 15:43:58 | INFO     | Curva TARGET_BPA_1.01/GradientBoosting: gap=0.037 diag=OK
2026-05-07 15:44:00 | INFO     | Curva TARGET_BPP_2.01/GradientBoosting: gap=0.052 diag=OK
2026-05-07 15:44:01 | INFO     | Curva TARGET_BPP_2.03/GradientBoosting: gap=0.125 diag=OK
2026-05-07 15:44:04 | INFO     | Curva TARGET_BPP_2/GradientBoosting: gap=0.037 diag=OK
2026-05-07 15:44:08 | INFO     | Curva TARGET_DFC_MI_6.01/RandomForest: gap=0.455 diag=overfitting


  ✅ Salvo: curvas_aprendizado.png


## Etapa 8 — Análise de Resíduos

In [10]:
print("\nGerando análise de resíduos...")

n_t = len(TARGETS)
fig, axes = plt.subplots(n_t, 2, figsize=(14, 5*n_t))
if n_t == 1: axes = axes.reshape(1, -1)

for i, target in enumerate(TARGETS):
    transformacao = get_target_transform(target)
    df_te = teste[FEATURES + [target]].copy()
    df_te = df_te[df_te[target].notna()]
    X_te  = df_te[FEATURES].values
    y_te  = df_te[target].values

    # Análise de resíduos usa o mesmo melhor modelo das outras etapas (R²)
    melhor_nome = melhores[target]
    mod    = resultados[target][melhor_nome][0]
    y_pred = target_inverse_transform(mod.predict(X_te), transformacao)
    residuos = y_te - y_pred

    # Predito × Observado
    ax1 = axes[i, 0]
    lim = max(np.nanmax(np.abs(y_te)), np.nanmax(np.abs(y_pred))) * 1.05
    ax1.scatter(y_pred, y_te, alpha=0.45, s=18, color='#1f4e79', edgecolors='none')
    ax1.plot([0, lim], [0, lim], 'r--', lw=1.5)
    ax1.set_xlabel('Predito (R$ mil)')
    ax1.set_ylabel('Observado (R$ mil)')
    ax1.set_title(f"{target.replace('TARGET_', '')} — {melhor_nome}\nPredito × Observado",
                   fontsize=10, fontweight='bold')
    r2_val = metricas_teste[target][melhor_nome]['R2_teste']
    ax1.text(0.05, 0.92, f'R²={r2_val:.3f}', transform=ax1.transAxes, fontsize=9,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # Resíduos × Predito
    ax2 = axes[i, 1]
    ax2.scatter(y_pred, residuos, alpha=0.45, s=18, color='#744210', edgecolors='none')
    ax2.axhline(0, color='r', lw=1.5, ls='--')
    ax2.axhline( np.std(residuos), color='gray', lw=1, ls=':', alpha=0.7)
    ax2.axhline(-np.std(residuos), color='gray', lw=1, ls=':', alpha=0.7)
    ax2.set_xlabel('Predito (R$ mil)')
    ax2.set_ylabel('Resíduo (R$ mil)')
    ax2.set_title(f'Resíduos  |  skew={pd.Series(residuos).skew():.2f}  '
                   f'σ={np.std(residuos):,.0f}', fontsize=10, fontweight='bold')

plt.suptitle('Análise de Resíduos — Conjunto de Teste 2023–2024',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'analise_residuos.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Salvo: analise_residuos.png")


Gerando análise de resíduos...
  ✅ Salvo: analise_residuos.png


## Etapa 9 — Persistência completa

In [11]:
rows_cv, rows_te = [], []

for target, algs in resultados.items():
    b = baselines.get(target, {})
    for alg, (_, m) in algs.items():
        rows_cv.append({
            'Target'       : target,
            'Algoritmo'    : alg,
            'RMSE_CV'      : m['RMSE_CV'],
            'RMSE_CV_std'  : m.get('RMSE_CV_std'),
            'SMAPE_CV'     : m['SMAPE_CV'],
            'R2_CV'        : m['R2_CV'],
            'TheilU_CV'    : m.get('TheilU_CV'),
            'DA_CV'        : m.get('DA_CV'),
            'transformacao': m.get('transformacao'),
            'log_transform': m['log_transform'],
            'best_params'  : str(m['best_params']),
        })
        mt = metricas_teste[target][alg]
        rows_te.append({
            'Target'         : target,
            'Algoritmo'      : alg,
            'RMSE_teste'     : mt['RMSE_teste'],
            'MAE_teste'      : mt['MAE_teste'],
            'SMAPE_teste'    : mt['SMAPE_teste'],
            'R2_teste'       : mt['R2_teste'],
            'TheilU_teste'   : mt.get('TheilU_teste'),
            'DA_teste'       : mt.get('DA_teste'),
            'RMSE_baseline'  : b.get('RMSE_baseline'),
            'Bateu_baseline' : mt['RMSE_teste'] < b.get('RMSE_baseline', np.inf),
            'TheilU_ok'      : (mt.get('TheilU_teste', 1.0) or 1.0) < 1.0,
        })

df_cv = pd.DataFrame(rows_cv)
df_te = pd.DataFrame(rows_te)

with open(PASTA_SAIDA / 'melhores_modelos.pkl', 'wb') as f:
    pickle.dump(melhores, f)

df_cv.to_csv(PASTA_SAIDA / 'resultados_cv.csv',    index=False)
df_te.to_csv(PASTA_SAIDA / 'resultados_teste.csv', index=False)

with open(PASTA_SAIDA / 'resultados_cv.pkl',       'wb') as f: pickle.dump(resultados, f)
with open(PASTA_SAIDA / 'metricas_teste.pkl',      'wb') as f: pickle.dump(metricas_teste, f)
with open(PASTA_SAIDA / 'baselines.pkl',           'wb') as f: pickle.dump(baselines, f)
with open(PASTA_SAIDA / 'feature_importances.pkl', 'wb') as f: pickle.dump(feature_importances, f)
with open(PASTA_SAIDA / 'melhores_modelos.pkl',    'wb') as f: pickle.dump(melhores, f)

# FIX: versão corrigida para 'V5_9targets_split_temporal'
relatorio = {
    'versao'            : 'V5_9targets_split_temporal',
    'ano_corte'         : ANO_CORTE,
    'n_treino'          : int(len(treino)),
    'n_teste'           : int(len(teste)),
    'algoritmos'        : list(ALGORITMOS.keys()),
    'targets'           : TARGETS,
    'log_targets'       : list(LOG_TARGETS),
    'arcsinh_targets'   : list(ARCSINH_TARGETS),
    'target_transforms' : {t: get_target_transform(t) for t in TARGETS},
    'features'          : FEATURES,
    'melhores'          : melhores,
    'baselines'         : {
        t: {k: float(v) for k, v in b.items()
            if isinstance(v, (int, float, np.floating))}
        for t, b in baselines.items()
    },
}
with open(PASTA_SAIDA / 'relatorio_modelagem.json', 'w', encoding='utf-8') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)

print("\n" + "═"*72)
print("  RESUMO FINAL — Script 3 V5 (baseline por empresa)")
print("═"*72)
print(f"  Treino  : {len(treino):,} obs (≤{ANO_CORTE}) | "
      f"DFP={(treino['ORIGEM']=='DFP').sum()} | ITR={(treino['ORIGEM']=='ITR').sum()}")
print(f"  Teste   : {len(teste):,} obs (≥{ANO_CORTE+1}) | "
      f"DFP={(teste['ORIGEM']=='DFP').sum()}  | ITR={(teste['ORIGEM']=='ITR').sum()}")
print(f"  Modelos : {len(TARGETS) * len(ALGORITMOS)} ({len(TARGETS)} targets × {len(ALGORITMOS)} algoritmos)")
print(f"  Targets : {len(TARGETS)} (3 primários + 5 balanço + 1 caixa)")
print(f"  Métricas: RMSE, MAE, SMAPE, R², Theil's U, Acurácia Direcional")
print(f"  Melhores por R²:")
for t, alg in melhores.items():
    m = metricas_teste[t][alg]
    b = baselines.get(t, {})
    bateu = m['RMSE_teste'] < b.get('RMSE_baseline', np.inf)
    print(f"    {t:<35} {alg:<20} R²={m['R2_teste']:.3f}  "
          f"RMSE={m['RMSE_teste']:,.0f}  "
          f"SMAPE={m['SMAPE_teste']:.1%}  U={m['TheilU_teste']:.3f}  "
          f"{'✅' if bateu else '❌'}")
print("═"*72)
print("  ✅ Pronto para Script 4 (Avaliação + Z'')")
print("═"*72)



════════════════════════════════════════════════════════════════════════
  RESUMO FINAL — Script 3 V5 (baseline por empresa)
════════════════════════════════════════════════════════════════════════
  Treino  : 713 obs (≤2022) | DFP=181 | ITR=532
  Teste   : 168 obs (≥2023) | DFP=24  | ITR=144
  Modelos : 36 (9 targets × 4 algoritmos)
  Targets : 9 (3 primários + 5 balanço + 1 caixa)
  Métricas: RMSE, MAE, SMAPE, R², Theil's U, Acurácia Direcional
  Melhores por R²:
    TARGET_DRE_3.01                     GradientBoosting     R²=0.996  RMSE=6,859,573  SMAPE=3.4%  U=0.125  ✅
    TARGET_DRE_3.11                     RandomForest         R²=0.826  RMSE=7,649,942  SMAPE=71.0%  U=0.594  ✅
    TARGET_EBITDA                       GradientBoosting     R²=0.840  RMSE=6,382,429  SMAPE=24.7%  U=0.938  ✅
    TARGET_BPA_1                        GradientBoosting     R²=0.976  RMSE=34,941,625  SMAPE=7.9%  U=0.287  ❌
    TARGET_BPA_1.01                     GradientBoosting     R²=0.980  RMSE=4,560,400 